# 학습데이터 증강&평가

### 계약서 증강

In [5]:
!pip install -q openai

In [6]:
import json, os, re, time, random
from google.colab import userdata
from openai import OpenAI

API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o"

In [8]:
# 1. 시드데이터 로드
with open('CON_SEED_DATA.json', encoding='utf-8') as f:
    seed_data = json.load(f)

print(f"시드 개수: {len(seed_data)}")

# 2. 카테고리별 chunk_pool 구성 (시드의 refs를 chunk_id 기준으로 중복 제거)
chunk_pool_by_category = {}
for item in seed_data:
    cat = item['category']
    pool = chunk_pool_by_category.setdefault(cat, {})
    for ref in item['refs']:
        pool[ref['chunk_id']] = {
            "chunk_id": ref['chunk_id'],
            "law_name": ref['law_name'],
            "article": ref['article'],
            "chunk_text": ref['chunk_text'],
        }

# 3. (선택) 별도 chunks.json이 있으면 병합 여부만 확인 (자동 병합은 하지 않음)
if os.path.exists('chunks.json'):
    with open('chunks.json', encoding='utf-8') as f:
        extra_chunks = json.load(f)
    print(f"chunks.json 발견: {len(extra_chunks)}개 조문 (카테고리 매핑은 수동 확인 필요, 자동 병합하지 않음)")
else:
    print("chunks.json 없음 — 시드데이터 refs 기반 chunk_pool만 사용")

for cat, pool in chunk_pool_by_category.items():
    print(f"- {cat}: chunk {len(pool)}개")

시드 개수: 30
chunks.json 없음 — 시드데이터 refs 기반 chunk_pool만 사용
- 지체상금: chunk 4개
- 대금지급: chunk 2개
- 검사: chunk 2개
- 하자담보: chunk 4개
- 이행보증: chunk 2개
- 계약해제·해지: chunk 2개
- 부당특약: chunk 2개
- 물가변동·조정: chunk 4개
- 과업범위: chunk 2개
- 지식재산권: chunk 4개
- 하도급: chunk 3개
- 장기계속·단가: chunk 2개


In [9]:
# 4. 전체 데이터셋 기준 기존 id 집합 (중복 방지)
existing_ids = set(item['id'] for item in seed_data)

# 5. 카테고리/패턴 목록
CATEGORIES = sorted(chunk_pool_by_category.keys())
PATTERNS = ["일치", "일치경계(B)", "불일치·을불리", "불일치·을유리", "검토(RAG miss)"]

TARGET_TOTAL = 150
BATCH_N = 5
MAX_CALLS = 60

print("카테고리:", CATEGORIES)

카테고리: ['검사', '계약해제·해지', '과업범위', '대금지급', '물가변동·조정', '부당특약', '이행보증', '장기계속·단가', '지식재산권', '지체상금', '하도급', '하자담보']


In [10]:
SYSTEM_PROMPT = """당신은 지방자치단체 계약서 조항 검토 학습데이터를 증강하는 생성기입니다.
목표: 기존 시드데이터와 동일한 스키마·품질 기준으로 새로운 예시를 생성한다.

[입력]
- category: 생성할 카테고리 (예: 지체상금, 대금지급, 검사, 하자담보, 이행보증, 계약해지, 부당특약, 물가변동, 과업범위, 지식재산권, 하도급, 장기계속 등)
- pattern: 일치 / 일치경계(B) / 불일치·을불리 / 불일치·을유리 / 검토(RAG miss) 중 하나
- chunk_pool: chunks.json에서 해당 category와 관련된 실제 조문 chunk 목록 (chunk_id, law_name, article_id, text)
- 기존 id 목록 (중복 방지용)

[생성 절차 — 반드시 순서대로]
1단계. refs 선정
- chunk_pool 안에서만 골라야 하며, text를 절대 요약·의역·재구성하지 않고 원문 그대로 chunk_text에 복사한다.
- pattern이 "검토(RAG miss)"인 경우: 카테고리와 관련은 있지만 판단 기준(수치·요건)은 담고 있지 않은 chunk를 의도적으로 고른다.
- pattern이 "일치경계(B)"인 경우: 주요 수치는 일치시키되, refs에 없는 세부조건(기간·예외 등)이 조항에 포함되도록 clause_text를 설계한다.

2단계. clause_text 작성
- 실제 계약서에서 쓰일 법한 조 형식(제○조(제목) 본문)으로 작성한다.
- 기존 시드데이터와 문장 구조·숫자가 겹치지 않게 다양화한다 (같은 패턴이라도 표현 방식, 조번호, 수치를 바꿀 것).
- pattern에 맞춰 수치를 조정한다:
  · 일치 → refs 기준과 동일한 수치/조건
  · 불일치→ 계약상대자에게 더 불리한 방향으로 기준을 벗어남 (예: 상한 축소, 요율 인상, 의무 가중)
  · 불일치→ 계약상대자에게 더 유리한 방향으로 기준을 벗어남 (예: 요율 인하, 기간 단축, 의무 면제)
  · 검토 → refs만으로는 옳고 그름을 가릴 수 없는 조항

3단계. label 부여
- 1·2단계에서 설계한 대로 기계적으로 결정한다 (임의 판단 금지).

4단계. 근거 작성 — 아래 규칙 그대로 적용
- 법률결론어(적법/위법/무효/유효/강행규정/부당 등) 사용 금지, 사실 대조 표현만 사용
- refs에 없는 조문·법령명을 새로 언급하지 않음 (특히 검토 패턴)
- 인용한 조문의 항/호 번호와 구체적 수치를 근거 문장에 명시해 비전문가가 원문 대조 가능하게 함
- 마지막에 고정 disclaimer 부착: "(참고용 검토이며 법적 판단 아님)" / 검토 패턴은 "(참고자료 부족에 따른 검토 보류이며 법적 판단 아님)"

5단계. 자체 검증 (생성 후 반드시 수행)
- refs의 chunk_text가 chunk_pool 원문과 글자 단위로 일치하는지 재확인
- 근거 문장에서 언급한 항/호 번호가 실제로 그 chunk_text 안에 존재하는지 재확인
- 위 두 조건 중 하나라도 어긋나면 해당 예시는 폐기하고 다시 생성

[판정규칙]
판정은 [일치/불일치/검토] 중 하나만 나오도록 한다

[출력 스키마]
반드시 아래 형태의 JSON 객체 하나만 반환한다. 다른 텍스트·설명·마크다운을 절대 포함하지 않는다.
{
  "items": [
    {
      "id": "{category}_{pattern요약}_{순번}",
      "category": "...",
      "pattern": "...",
      "clause_text": "...",
      "refs": [
        {
          "chunk_id": "...",
          "article": "...",
          "law_name": "...",
          "chunk_text": "..."
        }
      ],
      "label": "...",
      "근거": "..."
    }
  ]
}

[생성 개수]
- 1회 호출당 N개 (기본 5개), 서로 조번호·수치·표현이 겹치지 않게 생성"""

In [11]:
def build_user_message(category, pattern, chunk_pool, existing_ids, n=BATCH_N):
    payload = {
        "category": category,
        "pattern": pattern,
        "chunk_pool": list(chunk_pool.values()),
        "existing_ids": sorted(existing_ids),
        "n": n,
    }
    return json.dumps(payload, ensure_ascii=False)


def call_generator(category, pattern, chunk_pool, existing_ids, n=BATCH_N, retries=3):
    user_msg = build_user_message(category, pattern, chunk_pool, existing_ids, n)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0.7,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
            )
            raw = resp.choices[0].message.content
            parsed = json.loads(raw)
            items = parsed.get("items", [])
            if isinstance(items, list) and items:
                return items
        except Exception as e:
            print(f"  [warn] 호출 실패(attempt {attempt+1}): {e}")
            time.sleep(2)
    return []

In [12]:
# 자체 검증 로직
REQUIRED_KEYS = {"id", "category", "pattern", "clause_text", "refs", "label", "근거"}
DISCLAIMER_NORMAL = "(참고용 검토이며 법적 판단 아님)"
DISCLAIMER_REVIEW = "(참고자료 부족에 따른 검토 보류이며 법적 판단 아님)"

def validate_item(item, category, pattern, chunk_pool, existing_ids):
    if not REQUIRED_KEYS.issubset(item.keys()):
        return False, "필수 필드 누락"
    if item['id'] in existing_ids:
        return False, "id 중복"
    if item['category'] != category or item['pattern'] != pattern:
        return False, "category/pattern 불일치"
    if not item.get('refs'):
        return False, "refs 없음"

    for ref in item['refs']:
        cid = ref.get('chunk_id')
        if cid not in chunk_pool:
            return False, f"chunk_id({cid})가 chunk_pool에 없음"
        if ref.get('chunk_text') != chunk_pool[cid]['chunk_text']:
            return False, f"chunk_text가 원문과 불일치 ({cid})"

    geunge = item.get('근거', '')
    if pattern == "검토(RAG miss)":
        if DISCLAIMER_REVIEW not in geunge:
            return False, "검토 disclaimer 누락"
        if item.get('label') != '검토':
            return False, "검토 패턴인데 label이 검토가 아님"
    else:
        if DISCLAIMER_NORMAL not in geunge:
            return False, "일반 disclaimer 누락"

    banned_terms = ["위법하", "무효이", "강행규정", "적법하다", "유효하다고"]
    if any(t in geunge for t in banned_terms):
        return False, "법률결론어 사용"

    return True, "ok"

In [13]:
# 메인 생성 루프
augmented = []
call_count = 0

def current_total():
    return len(seed_data) + len(augmented)

while current_total() < TARGET_TOTAL and call_count < MAX_CALLS:
    for category in CATEGORIES:
        if current_total() >= TARGET_TOTAL or call_count >= MAX_CALLS:
            break
        pattern = PATTERNS[call_count % len(PATTERNS)]
        pool = chunk_pool_by_category.get(category, {})
        if not pool:
            continue

        remaining = TARGET_TOTAL - current_total()
        n = min(BATCH_N, remaining)
        print(f"[{call_count+1}] {category} / {pattern} / 요청 {n}개 (누적 {current_total()}/{TARGET_TOTAL})")

        items = call_generator(category, pattern, pool, existing_ids, n=n)
        call_count += 1

        accepted = 0
        for item in items:
            ok, reason = validate_item(item, category, pattern, pool, existing_ids)
            if ok:
                augmented.append(item)
                existing_ids.add(item['id'])
                accepted += 1
            else:
                print(f"    폐기: {item.get('id','?')} - {reason}")
        print(f"    -> {accepted}/{len(items)}개 채택")

        with open('augmented_progress.json', 'w', encoding='utf-8') as f:
            json.dump(augmented, f, ensure_ascii=False, indent=2)

        time.sleep(1)

print(f"\n최종 생성: {len(augmented)}개 (전체 {current_total()}/{TARGET_TOTAL})")

[1] 검사 / 일치 / 요청 5개 (누적 30/150)
    -> 5/5개 채택
[2] 계약해제·해지 / 일치경계(B) / 요청 5개 (누적 35/150)
    -> 5/5개 채택
[3] 과업범위 / 불일치·을불리 / 요청 5개 (누적 40/150)
    -> 5/5개 채택
[4] 대금지급 / 불일치·을유리 / 요청 5개 (누적 45/150)
    -> 5/5개 채택
[5] 물가변동·조정 / 검토(RAG miss) / 요청 5개 (누적 50/150)
    -> 5/5개 채택
[6] 부당특약 / 일치 / 요청 5개 (누적 55/150)
    -> 5/5개 채택
[7] 이행보증 / 일치경계(B) / 요청 5개 (누적 60/150)
    -> 5/5개 채택
[8] 장기계속·단가 / 불일치·을불리 / 요청 5개 (누적 65/150)
    -> 5/5개 채택
[9] 지식재산권 / 불일치·을유리 / 요청 5개 (누적 70/150)
    -> 5/5개 채택
[10] 지체상금 / 검토(RAG miss) / 요청 5개 (누적 75/150)
    -> 5/5개 채택
[11] 하도급 / 일치 / 요청 5개 (누적 80/150)
    -> 5/5개 채택
[12] 하자담보 / 일치경계(B) / 요청 5개 (누적 85/150)
    -> 5/5개 채택
[13] 검사 / 불일치·을불리 / 요청 5개 (누적 90/150)
    -> 5/5개 채택
[14] 계약해제·해지 / 불일치·을유리 / 요청 5개 (누적 95/150)
    -> 5/5개 채택
[15] 과업범위 / 검토(RAG miss) / 요청 5개 (누적 100/150)
    -> 5/5개 채택
[16] 대금지급 / 일치 / 요청 5개 (누적 105/150)
    -> 5/5개 채택
[17] 물가변동·조정 / 일치경계(B) / 요청 5개 (누적 110/150)
    -> 5/5개 채택
[18] 부당특약 / 불일치·을불리 / 요청 5개 (누적 115/150)
    -> 5/5개 채택
[19] 이행보증

In [15]:
# 최종 저장
final_dataset = seed_data + augmented
with open('CON_AUGMENTED_150.json', 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=2)

print(f"최종 데이터셋 크기: {len(final_dataset)}개")
from collections import Counter
print("카테고리 분포:", Counter(d['category'] for d in final_dataset))
print("패턴 분포:", Counter(d['pattern'] for d in final_dataset))
print("label 분포:", Counter(d['label'] for d in final_dataset))

최종 데이터셋 크기: 150개
카테고리 분포: Counter({'하자담보': 15, '지체상금': 14, '대금지급': 13, '계약해제·해지': 13, '부당특약': 13, '검사': 12, '이행보증': 12, '물가변동·조정': 12, '지식재산권': 12, '하도급': 12, '과업범위': 11, '장기계속·단가': 11})
패턴 분포: Counter({'일치': 30, '일치경계(B)': 30, '불일치·을불리': 25, '불일치·을유리': 25, '검토(RAG miss)': 24, '불일치·을불리·A': 5, '불일치·을불리·B': 4, '불일치·을유리·A': 3, '불일치·을유리·B': 2, '불일치': 2})
label 분포: Counter({'일치': 35, '일치경계': 25, '불일치·을불리': 25, '을유리': 25, '검토': 24, '불일치': 16})
